# Evaluación final del champion en test — #22
El champion se eligió en `08_comparison.ipynb` con la regla acordada (mejor F1 macro medio de validación con gap < 0,05): **XGBoost**.

Este notebook hace **la única evaluación en el test reservado**. Para que se haga una sola vez y de forma limpia:
1. **Antes de tocar test** comprueba que el champion, sus parámetros, el split y los umbrales coinciden con los registrados en la comparación. Si algo no coincide, se detiene sin predecir.
2. Ajusta el Pipeline del champion (`src/xgboost_model.py`, prueba seleccionada por Optuna) con **todo train**.
3. Predice test **una vez**, calcula las métricas acordadas (accuracy, F1 macro, precision/recall/F1 por clase, matriz de confusión, errores extremos) y **las guarda en `docs/results/champion_test.json` antes de mostrarlas**.
4. Si `champion_test.json` ya existe, el notebook se detiene al principio: no se repite la evaluación.

`tests/test_champion_notebook.py` ejecuta este notebook completo con datos sintéticos para comprobar el código sin consumir el test real.

Después de ver estos resultados no se cambia el modelo, las variables ni los hiperparámetros. **Limitación:** la baseline de regresión logística mostró en el pasado resultados sobre esta misma partición (exposición histórica, ver `docs/decisiones.md`); no guió la elección del champion, pero se declara.

In [ ]:
import hashlib, json, platform, subprocess, sys
from datetime import date
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import sklearn, xgboost
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from src.cleaning import MODEL_FEATURES
from src.preprocessing import RAW_DATA_PATH, TARGET_LABELS, load_train_test
from src.xgboost_model import FIXED_PARAMS, encode_target, make_pipeline

RESULTS = ROOT / 'docs/results'
OUTPUT = RESULTS / 'champion_test.json'
if OUTPUT.exists():
    raise RuntimeError(f'{OUTPUT.name} ya existe: el test ya se evaluó una vez. No se repite.')

# Champion por la regla acordada, recalculado desde las tablas de cada modelo
results = pd.concat([pd.read_csv(p) for p in sorted(RESULTS.glob('*_comparison.csv'))])
candidates = results[(results['model'] != 'DummyClassifier') & (results['gap'] < 0.05)]
champion_row = candidates.sort_values('cv_validation_f1_macro', ascending=False).iloc[0]
assert champion_row['model'] == 'XGBoost', f"Champion inesperado: {champion_row['model']}"

details = json.loads((RESULTS / 'xgboost_details.json').read_text())
params = details['selected_params']
assert not details['test_evaluated'], 'xgboost_details.json indica que el test ya se evaluó'

raw_sha256 = hashlib.sha256(RAW_DATA_PATH.read_bytes()).hexdigest()
assert raw_sha256 == champion_row['raw_sha256'] == details['provenance']['raw_sha256'], \
    'El archivo de datos no es el mismo que se usó en la comparación'
print('Champion:', champion_row['model'], '· prueba Optuna', details['selected_trial'])
params

## 1. Ajuste con todo train
Todavía no se mira test: solo se comprueba que el split es el mismo que el de la comparación.

In [ ]:
X_train, X_test, y_train_labels, y_test_labels, thresholds = load_train_test(return_thresholds=True)

# Mismo split y mismos umbrales que cuando se eligió el champion
assert list(X_train.columns) == list(X_test.columns) == list(MODEL_FEATURES)
assert len(X_train) == details['train_rows'], 'Train no coincide con el de la comparación'
assert all(np.isclose(thresholds[k], details['thresholds'][k]) for k in ('lower', 'upper')), \
    'Los umbrales no coinciden con los de la comparación'
assert X_train.index.intersection(X_test.index).empty, 'Train y test comparten filas'

y_train = encode_target(y_train_labels)          # Económico=0, Medio=1, Premium=2
champion = make_pipeline(params).fit(X_train, y_train)
print('Train:', X_train.shape, '· Test:', X_test.shape, '· Umbrales de train:', thresholds)

## 2. Evaluación única en test
Predicción, métricas y registro en la misma celda; el archivo se guarda **antes** de mostrar nada.

In [ ]:
INT_TO_LABEL = dict(enumerate(TARGET_LABELS))
labels = list(TARGET_LABELS)
y_pred = pd.Series(champion.predict(X_test), index=X_test.index).map(INT_TO_LABEL)
y_true = y_test_labels.astype(object)
assert y_pred.notna().all() and len(y_pred) == len(y_true)

matrix = confusion_matrix(y_true, y_pred, labels=labels)
test_metrics = {
    'accuracy': accuracy_score(y_true, y_pred),
    'f1_macro': f1_score(y_true, y_pred, average='macro'),
    'extreme_error_rate': (matrix[0, 2] + matrix[2, 0]) / matrix.sum(),
}
report = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)

record = {
    'issue': 22,
    'champion': 'XGBoost',
    'selection_rule': 'max mean validation macro F1 among models with gap < 0.05 (08_comparison.ipynb)',
    'selected_trial': details['selected_trial'],
    'params': {**FIXED_PARAMS, **params},
    'fit_on': 'full train',
    'evaluation_set': 'held_out_test',
    'evaluated_once': True,
    'historical_test_exposure': 'La baseline de regresión logística mostró antes resultados de esta partición; no guió la elección del champion.',
    'n_train': len(X_train),
    'n_test': len(X_test),
    'thresholds': thresholds,
    'cv': {
        'validation_f1_macro': champion_row['cv_validation_f1_macro'],
        'validation_std': champion_row['cv_validation_std'],
        'oof_accuracy': champion_row['oof_accuracy'],
        'oof_extreme_error_rate': champion_row['oof_extreme_error_rate'],
    },
    'test': test_metrics,
    'confusion_matrix': matrix.tolist(),
    'matrix_rows': 'real price class',
    'matrix_columns': 'predicted class',
    'label_order': labels,
    'classification_report': report,
    'provenance': {
        'date': date.today().isoformat(),
        'snapshot': champion_row['snapshot'],
        'raw_sha256': raw_sha256,
        'code_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip(),
        'python': platform.python_version(),
        'versions': {'pandas': pd.__version__, 'numpy': np.__version__,
                     'scikit-learn': sklearn.__version__, 'xgboost': xgboost.__version__},
    },
}
OUTPUT.write_text(json.dumps(record, indent=2, ensure_ascii=False, default=float) + '\n')
print('Guardado:', OUTPUT.relative_to(ROOT))

## 3. Resultados
Solo lectura de lo ya guardado: si esta celda fallara, se puede repetir sin volver a predecir test.

In [ ]:
saved = json.loads(OUTPUT.read_text())

comparison = pd.DataFrame({
    'Validación cruzada (train)': [saved['cv']['validation_f1_macro'], saved['cv']['oof_accuracy'],
                                   saved['cv']['oof_extreme_error_rate']],
    'Test (una vez)': [saved['test']['f1_macro'], saved['test']['accuracy'],
                       saved['test']['extreme_error_rate']],
}, index=['F1 macro', 'Accuracy', 'Errores extremos'])
display(comparison.round(4))

per_class = pd.DataFrame(saved['classification_report']).T.loc[saved['label_order'],
                                                               ['precision', 'recall', 'f1-score', 'support']]
display(per_class.round(3))

matrix_table = pd.DataFrame(
    saved['confusion_matrix'],
    index=pd.Index(saved['label_order'], name='Categoría REAL (según precio)'),
    columns=pd.Index(saved['label_order'], name='Categoría PREDICHA por el modelo'),
)
display(matrix_table)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay(np.array(saved['confusion_matrix']), display_labels=saved['label_order']).plot(
    cmap='Blues', colorbar=False)
plt.xlabel('Categoría predicha por el modelo')
plt.ylabel('Categoría real (según precio y umbrales de train)')
plt.title('XGBoost · test reservado (evaluación única)')
plt.show()

## Conclusiones
*(Se completan después de la evaluación única.)*